In [ ]:
import os

# 1. Set the download directory BEFORE importing FiftyOne
# This stops it from downloading to ~/fiftyone or /root/fiftyone
LOCAL_ZOO_DIR = os.path.join(os.getcwd(), "fiftyone_data")
os.environ["FIFTYONE_DATASET_ZOO_DIR"] = LOCAL_ZOO_DIR

import json
import shutil
import fiftyone.zoo as foz
from PIL import Image

# Configuration variables (adjust as needed)
SAMPLE_SIZE = 5000
REPO_ROOT  = os.path.abspath(os.path.join(os.getcwd(), ".."))
IMAGES_DIR = os.path.join(REPO_ROOT, "data", "gen", "raw_images")
JSON_PATH  = os.path.join(REPO_ROOT, "data", "gen", "fosae_labeled_dataset_downloaded.json")

try:
    print(f"Dataset will be downloaded/loaded from: {LOCAL_ZOO_DIR}")
    print(f"Images  -> {IMAGES_DIR}")
    print(f"JSON    -> {JSON_PATH}")

    dataset = foz.load_zoo_dataset(
        "coco-2017",
        split="train",
        label_types=["detections"],   # only object boxes; skip segmentation masks
        max_samples=SAMPLE_SIZE,
        seed=42,
        shuffle=True,
    )
    print(f"Loaded {len(dataset)} samples")

    os.makedirs(IMAGES_DIR, exist_ok=True)
    annotations = []

    for sample in dataset:
        filename = os.path.basename(sample.filepath)
        shutil.copy(sample.filepath, os.path.join(IMAGES_DIR, filename))

        with Image.open(sample.filepath) as img:
            width, height = img.size

        objects = []
        detections = sample.ground_truth

        if detections and detections.detections:
            for det in detections.detections:
                if det.bounding_box:   # [x_norm, y_norm, w_norm, h_norm] top-left + dims
                    bb = det.bounding_box
                    objects.append({
                        "class": det.label,
                        "bbox": [
                            int(bb[0] * width),
                            int(bb[1] * height),
                            int((bb[0] + bb[2]) * width),
                            int((bb[1] + bb[3]) * height),
                        ],
                    })

        # skip images with no annotations
        if objects:
            annotations.append({"image": filename, "objects": objects})

    with open(JSON_PATH, "w") as f:
        json.dump(annotations, f, indent=2)

    print(f"Done: {len(annotations)} annotated images")
    print(f"Images -> {IMAGES_DIR}/")
    print(f"JSON   -> {JSON_PATH}")
    print()
    print("To train FOSAE run:")
    print(f"  cd {REPO_ROOT}")
    print("  python strips.py learn labeled_objects FirstOrderSAE None None None None None None None all_pairs")

except Exception as e:
    print("Execution failed:", e)


Dataset will be downloaded/loaded from: /home/panoslat/Dev/FOSAE/labeled-fosae/notebooks/fiftyone_data
Images  -> /home/panoslat/Dev/FOSAE/labeled-fosae/data/gen/raw_images
JSON    -> /home/panoslat/Dev/FOSAE/labeled-fosae/data/gen/fosae_labeled_dataset_unsloth.json
 100% |██████|    1.9Gb/1.9Gb [34.5s elapsed, 0s remaining, 121.7Mb/s]      
Extracting annotations to '/home/panoslat/Dev/FOSAE/labeled-fosae/notebooks/fiftyone_data/coco-2017/raw/instances_train2017.json'
 100% |████████████████| 5000/5000 [9.0m elapsed, 0s remaining, 11.4 images/s]      
Writing annotations for 5000 downloaded samples to '/home/panoslat/Dev/FOSAE/labeled-fosae/notebooks/fiftyone_data/coco-2017/train/labels.json'
Dataset info written to '/home/panoslat/Dev/FOSAE/labeled-fosae/notebooks/fiftyone_data/coco-2017/info.json'
Loading 'coco-2017' split 'train'
 100% |███████████████| 5000/5000 [1.1m elapsed, 0s remaining, 77.8 samples/s]       
Dataset 'coco-2017-train-5000' created
Loaded 5000 samples
Done: 495